In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

TRAIN_SZ = 0.8
VALID_SZ = 1.0 - TRAIN_SZ

# Load data
data = pd.read_csv("../datasets/melbourne-housing-snapshot/melb_data.csv")

# Separate predictors and target
y = data.Price
X = data.drop(['Price'], axis=1)

# Divide data into training and validation subsets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, 
                                                      train_size=TRAIN_SZ, test_size=VALID_SZ,
                                                      random_state=0)

In [2]:
X_train.head()

,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
12167,St Kilda,11/22 Charnwood Cr,1,u,S,hockingstuart,29/07/2017,5.0,3182.0,1.0,1.0,1.0,0.0,NaN,1940.0,Port Phillip,-37.85984,144.9867,Southern Metropolitan,13240.0
6524,Williamstown,18 James St,2,h,SA,Hunter,17/09/2016,8.0,3016.0,2.0,2.0,1.0,193.0,NaN,NaN,Hobsons Bay,-37.85800,144.9005,Western Metropolitan,6380.0
8413,Sunshine,10 Dundalk St,3,h,S,Barry,8/04/2017,12.6,3020.0,3.0,1.0,1.0,555.0,NaN,NaN,Brimbank,-37.79880,144.8220,Western Metropolitan,3755.0
2919,Glenroy,1/2 Prospect St,3,u,SP,Brad,18/06/2016,13.0,3046.0,3.0,1.0,1.0,265.0,NaN,1995.0,Moreland,-37.70830,144.9158,Northern Metropolitan,8870.0
6043,Sunshine North,35 Furlong Rd,3,h,S,First,22/05/2016,13.3,3020.0,3.0,1.0,2.0,673.0,673.0,1970.0,Brimbank,-37.76230,144.8272,Western Metropolitan,4217.0


In [3]:
X_valid.head()

,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
8505,Williamstown,44 Electra St,4,h,SP,Greg,6/05/2017,8.0,3016.0,4.0,2.0,2.0,450.0,190.0,1910.0,Hobsons Bay,-37.86100,144.89850,Western Metropolitan,6380.0
5523,Seddon,80 Gamon St,2,h,S,Chisholm,30/07/2016,6.6,3011.0,2.0,1.0,0.0,172.0,81.0,1900.0,Maribyrnong,-37.81000,144.88960,Western Metropolitan,2417.0
12852,Sunshine North,6 Melton Av,3,h,SP,Sweeney,16/09/2017,10.5,3020.0,3.0,1.0,1.0,581.0,NaN,NaN,NaN,-37.76740,144.82421,Western Metropolitan,4217.0
4818,Prahran,16 Park Rd,3,t,PI,Marshall,6/08/2016,4.5,3181.0,2.0,2.0,1.0,128.0,134.0,2000.0,Stonnington,-37.85260,145.00710,Southern Metropolitan,7717.0
12812,Pascoe Vale,13 Yorkshire St,3,h,S,Nelson,16/09/2017,8.5,3044.0,3.0,2.0,2.0,480.0,NaN,NaN,NaN,-37.72523,144.94567,Northern Metropolitan,7485.0


In [4]:
# Select categorical columns with relatively low cardinality
N = 10
categorical_cols = [col for col in X.columns 
                    if X[col].dtype == "object" 
                    and X[col].nunique() < N]
# Select numerical columns
numerical_cols = [col for col in X.columns 
                  if X[col].dtype in ["int64", "float64"]]

# Keep selected columns only
predictor_cols = categorical_cols + numerical_cols
X_train = X_train[predictor_cols]
X_valid = X_valid[predictor_cols]

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='constant')

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [6]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=0)

In [7]:
from sklearn.metrics import mean_absolute_error

# Bundle preprocessing and modeling code in a pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Preprocessing of training data, fit model 
pipeline.fit(X_train, y_train)

# Preprocessing of validation data, get predictions
y_pred = pipeline.predict(X_valid)

# Evaluate the model
mae = mean_absolute_error(y_valid, y_pred)
print('MAE:', mae)

MAE: 160679.18917034855
